# Autoregressive Transformer No-Context Forecasting

<!-- Commented sorted notebook -->
Purpose: train and evaluate the cleaned supervisor-facing version of this generative forecaster. This version receives known future/static covariates only; the past kWh context is intentionally removed.

The notebook follows the shared experiment structure: setup, preprocessing, batching, model training, checkpointing, validation sampling, and a printed metric table.


## Implementation And Metric References

<!-- Implementation reference map -->

This notebook contains project-specific code, but the main patterns are deliberately aligned with recognised libraries and public implementations so the workflow can be audited.

Common data/preprocessing references:

- pandas time-series resampling/dataframe operations: https://github.com/pandas-dev/pandas
- NumPy array operations: https://github.com/numpy/numpy
- scikit-learn preprocessing and splitting utilities: https://github.com/scikit-learn/scikit-learn
- PyTorch tensor, Dataset, DataLoader, optimiser, and neural-network building blocks: https://github.com/pytorch/pytorch

Shared metric/evaluation references:

- MAE/RMSE-style regression metrics: https://github.com/scikit-learn/scikit-learn/blob/main/sklearn/metrics/_regression.py
- Dynamic Time Warping implementation reference: https://github.com/tslearn-team/tslearn/blob/main/tslearn/metrics/dtw_variants.py
- KL/entropy implementation reference: https://github.com/scipy/scipy/blob/main/scipy/stats/_entropy.py
- Autocorrelation implementation reference: https://github.com/statsmodels/statsmodels/blob/main/statsmodels/tsa/stattools/_stattools.py
- Fréchet/FID-style feature distance implementation reference: https://github.com/mseitzer/pytorch-fid
- Proper scoring rules, interval scores, CRPS and quantile scoring reference: https://github.com/epiforecasts/scoringutils
- Minimal xlsx-writing idea/reference implementation: https://github.com/ericgazoni/openpyxl

Project-specific parts:

- The exact customer split, manually selected context/target lengths, selected covariates, composite score, and printed metric table format are thesis-specific decisions made for this experiment.

Model-specific implementation references:

- minGPT for compact causal Transformer/GPT-style masking and autoregressive sampling: https://github.com/karpathy/minGPT
- nanoGPT for a larger but still readable causal Transformer implementation: https://github.com/karpathy/nanoGPT
- PyTorch Transformer sequence modelling reference: https://github.com/pytorch/examples/tree/main/word_language_model


In [ ]:

import math
import os
import random
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
# scikit-learn scalers are used for standard preprocessing; implementation: https://github.com/scikit-learn/scikit-learn/tree/main/sklearn/preprocessing
from sklearn.preprocessing import MinMaxScaler, StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
# PyTorch Dataset/DataLoader is the standard minibatch input pipeline; implementation: https://github.com/pytorch/pytorch/tree/main/torch/utils/data
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore", message=r".*'H' is deprecated.*", category=FutureWarning)

MODEL_LABEL = "Autoregressive-Transformer-NoContext-Forecaster"
MODEL_TAG = "forecast-nocontext-autoregressive-transformer"

HOUSEHOLD_ID_COL = "CUSTOMER_KEY"
TIME_COL = "READING_DATETIME"
TARGET_COL = "kWh"

# Manual window length settings.
# FREQ only controls data aggregation/resampling. SEQ_LEN is edited directly by the user.
# No-context generative runs intentionally keep CONTEXT_LEN = 0.
# Reference target lengths:
#   30min: 24h=48, 7d=336, 28d=1344
#   1H:    24h=24, 7d=168, 28d=672
#   2H:    24h=12, 7d=84,  28d=336
#   3H:    24h=8,  7d=56,  28d=224
#   6H:    24h=4,  7d=28,  28d=112
FREQ = "30min"
PRIMARY_HORIZON = "24h"  # descriptive label only; edit SEQ_LEN manually when changing this label.
SEQ_LEN = 48
CONTEXT_LEN = 0
WINDOW_STRIDE = 1

SEED = 0
VAL_FRAC = 0.20
BATCH_SIZE = 64
LR = 2e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


epochs_ar = 500
hidden_dim = 128
cond_dim = 32
latent_dim = 32
grad_clip = 1.0



transformer_layers = 2
transformer_heads = 4
transformer_ff_dim = 256
transformer_dropout = 0.10

min_log_scale = -6.0
max_log_scale = 0.5
sample_temperature = 0.8



STATIC_COLS = [
    "NUM_OCCUPANTS", "NUM_ROOMS_HEATED", "NUM_REFRIGERATORS",
    "Unit", "SemiDetached", "SeparateHouse",
    "HAS_GAS_HEATING", "HAS_GAS_HOT_WATER", "HAS_GAS_COOKING",
    "HAS_POOLPUMP", "Ducted", "SplitSystem", "NoAirCon", "OtherAirCon",
    "CONTROLLED_LOAD_CNT",
]

CATEGORICAL_COLS = ["StationNo", "TRIAL_REGION_NAME"]

# Past covariates are only observed over the historical context window.
# Both lag-name spellings are listed because earlier exports used inconsistent column casing.
PAST_COV_COLS = [
    "Temperature",
    "CDD", "HDD",
    "wind_speed",
    "Temperature_lag_2", "Temperature_lag_6", "Temperature_lag_48", "Temperature_lag_144",
    "temperature_lag_2", "temperature_lag_6", "temperature_lag_48", "temperature_lag_144",
]

# Future covariates must be knowable before inference, so they are calendar/cyclical only.
FUTURE_COV_COLS = [
    "Hour_sin", "Hour_cos",
    "Weekday_sin", "Weekday_cos",
    "Month_sin", "Month_cos",
]

RAW_TIME_COV_COLS = [
    "Temperature",
    "Weekday",
    "Month",
    "Hour",
    "CDD",
    "HDD",
    "wind_speed",
    "Temperature_lag_2", "Temperature_lag_6", "Temperature_lag_48", "Temperature_lag_144",
    "temperature_lag_2", "temperature_lag_6", "temperature_lag_48", "temperature_lag_144",
]

# Keeps pandas frequency strings explicit so 30min/1H/2H/3H experiments resample consistently.
def pandas_freq(freq: str) -> str:
    return freq.replace("H", "h")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.set_float32_matmul_precision("medium")

print(f"{MODEL_LABEL} | FREQ={FREQ} | label={PRIMARY_HORIZON} | context_steps={CONTEXT_LEN} | target_steps={SEQ_LEN} | stride={WINDOW_STRIDE} | device={DEVICE}")


### Preprocessing Roadmap

The preprocessing stage is intentionally a short pipeline rather than a model-specific trick:

1. Load the shared household energy table.
2. Keep the columns needed for this experiment and fill any missing optional covariates.
3. Convert timestamps, customer IDs, numeric covariates, and categorical IDs into model-ready types.
4. Add cyclical calendar features so hour, weekday, and month wrap around naturally.
5. Resample each customer to the selected frequency.
6. Split by customer so validation households are unseen during training.
7. Build context/target windows or library time-series objects, then scale the model inputs.

The small checks in this section are kept only where they prevent silent data leakage, missing-column errors, or invalid tensor shapes.


In [ ]:

# The supervisor-facing notebooks expect the dataset pickle in the previous directory.
data_path = Path("..") / "data_with_weather.pickle"
df = pd.read_pickle(data_path)

# Cyclical sin/cos encoding preserves periodic hour/weekday/month distance; source: https://feature-engine.trainindata.com/en/latest/user_guide/creation/CyclicalFeatures.html
def add_cyclical_time_features(frame, datetime_values):
    dt = pd.DatetimeIndex(pd.to_datetime(datetime_values))
    hour = dt.hour.astype(np.float32)
    minute = dt.minute.astype(np.float32)
    hour_float = hour + minute / 60.0
    weekday = dt.weekday.astype(np.float32)
    month = dt.month.astype(np.float32)

    frame["Hour_sin"] = np.sin(2 * np.pi * hour_float / 24.0).astype("float32")
    frame["Hour_cos"] = np.cos(2 * np.pi * hour_float / 24.0).astype("float32")
    frame["Weekday_sin"] = np.sin(2 * np.pi * weekday / 7.0).astype("float32")
    frame["Weekday_cos"] = np.cos(2 * np.pi * weekday / 7.0).astype("float32")
    frame["Month_sin"] = np.sin(2 * np.pi * (month - 1.0) / 12.0).astype("float32")
    frame["Month_cos"] = np.cos(2 * np.pi * (month - 1.0) / 12.0).astype("float32")
    frame["Hour"] = hour.astype("float32")
    frame["Weekday"] = weekday.astype("float32")
    frame["Month"] = month.astype("float32")
    return frame

# Keep only expected columns. Missing optional covariates are created as zeros so the notebook shape stays stable.
if "READING_DATETIME" not in df.columns: df["READING_DATETIME"] = 0.0
if "kWh" not in df.columns: df["kWh"] = 0.0
if "CUSTOMER_KEY" not in df.columns: df["CUSTOMER_KEY"] = 0.0
if "NUM_OCCUPANTS" not in df.columns: df["NUM_OCCUPANTS"] = 0.0
if "NUM_ROOMS_HEATED" not in df.columns: df["NUM_ROOMS_HEATED"] = 0.0
if "NUM_REFRIGERATORS" not in df.columns: df["NUM_REFRIGERATORS"] = 0.0
if "Unit" not in df.columns: df["Unit"] = 0.0
if "SemiDetached" not in df.columns: df["SemiDetached"] = 0.0
if "SeparateHouse" not in df.columns: df["SeparateHouse"] = 0.0
if "HAS_GAS_HEATING" not in df.columns: df["HAS_GAS_HEATING"] = 0.0
if "HAS_GAS_HOT_WATER" not in df.columns: df["HAS_GAS_HOT_WATER"] = 0.0
if "HAS_GAS_COOKING" not in df.columns: df["HAS_GAS_COOKING"] = 0.0
if "HAS_POOLPUMP" not in df.columns: df["HAS_POOLPUMP"] = 0.0
if "Ducted" not in df.columns: df["Ducted"] = 0.0
if "SplitSystem" not in df.columns: df["SplitSystem"] = 0.0
if "NoAirCon" not in df.columns: df["NoAirCon"] = 0.0
if "OtherAirCon" not in df.columns: df["OtherAirCon"] = 0.0
if "CONTROLLED_LOAD_CNT" not in df.columns: df["CONTROLLED_LOAD_CNT"] = 0.0
if "StationNo" not in df.columns: df["StationNo"] = "missing"
if "TRIAL_REGION_NAME" not in df.columns: df["TRIAL_REGION_NAME"] = "missing"
if "Temperature" not in df.columns: df["Temperature"] = 0.0
if "Weekday" not in df.columns: df["Weekday"] = 0.0
if "Month" not in df.columns: df["Month"] = 0.0
if "Hour" not in df.columns: df["Hour"] = 0.0
if "CDD" not in df.columns: df["CDD"] = 0.0
if "HDD" not in df.columns: df["HDD"] = 0.0
if "wind_speed" not in df.columns: df["wind_speed"] = 0.0
if "Temperature_lag_2" not in df.columns: df["Temperature_lag_2"] = 0.0
if "Temperature_lag_6" not in df.columns: df["Temperature_lag_6"] = 0.0
if "Temperature_lag_48" not in df.columns: df["Temperature_lag_48"] = 0.0
if "Temperature_lag_144" not in df.columns: df["Temperature_lag_144"] = 0.0
if "temperature_lag_2" not in df.columns: df["temperature_lag_2"] = 0.0
if "temperature_lag_6" not in df.columns: df["temperature_lag_6"] = 0.0
if "temperature_lag_48" not in df.columns: df["temperature_lag_48"] = 0.0
if "temperature_lag_144" not in df.columns: df["temperature_lag_144"] = 0.0
needed = [TIME_COL, TARGET_COL, HOUSEHOLD_ID_COL] + STATIC_COLS + CATEGORICAL_COLS + RAW_TIME_COV_COLS
df = df[needed].copy()

# Convert identifiers and timestamps before sorting.
df[TIME_COL] = pd.to_datetime(df[TIME_COL])
df[HOUSEHOLD_ID_COL] = pd.to_numeric(df[HOUSEHOLD_ID_COL], errors="coerce").astype("int64")
df = df.sort_values([HOUSEHOLD_ID_COL, TIME_COL])

# Convert each continuous covariate explicitly so the preprocessing reads column-by-column.
df["NUM_OCCUPANTS"] = pd.to_numeric(df["NUM_OCCUPANTS"], errors="coerce").fillna(0.0).astype("float32")
df["NUM_ROOMS_HEATED"] = pd.to_numeric(df["NUM_ROOMS_HEATED"], errors="coerce").fillna(0.0).astype("float32")
df["NUM_REFRIGERATORS"] = pd.to_numeric(df["NUM_REFRIGERATORS"], errors="coerce").fillna(0.0).astype("float32")
df["Unit"] = pd.to_numeric(df["Unit"], errors="coerce").fillna(0.0).astype("float32")
df["SemiDetached"] = pd.to_numeric(df["SemiDetached"], errors="coerce").fillna(0.0).astype("float32")
df["SeparateHouse"] = pd.to_numeric(df["SeparateHouse"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_GAS_HEATING"] = pd.to_numeric(df["HAS_GAS_HEATING"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_GAS_HOT_WATER"] = pd.to_numeric(df["HAS_GAS_HOT_WATER"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_GAS_COOKING"] = pd.to_numeric(df["HAS_GAS_COOKING"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_POOLPUMP"] = pd.to_numeric(df["HAS_POOLPUMP"], errors="coerce").fillna(0.0).astype("float32")
df["Ducted"] = pd.to_numeric(df["Ducted"], errors="coerce").fillna(0.0).astype("float32")
df["SplitSystem"] = pd.to_numeric(df["SplitSystem"], errors="coerce").fillna(0.0).astype("float32")
df["NoAirCon"] = pd.to_numeric(df["NoAirCon"], errors="coerce").fillna(0.0).astype("float32")
df["OtherAirCon"] = pd.to_numeric(df["OtherAirCon"], errors="coerce").fillna(0.0).astype("float32")
df["CONTROLLED_LOAD_CNT"] = pd.to_numeric(df["CONTROLLED_LOAD_CNT"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature"] = pd.to_numeric(df["Temperature"], errors="coerce").fillna(0.0).astype("float32")
df["Weekday"] = pd.to_numeric(df["Weekday"], errors="coerce").fillna(0.0).astype("float32")
df["Month"] = pd.to_numeric(df["Month"], errors="coerce").fillna(0.0).astype("float32")
df["Hour"] = pd.to_numeric(df["Hour"], errors="coerce").fillna(0.0).astype("float32")
df["CDD"] = pd.to_numeric(df["CDD"], errors="coerce").fillna(0.0).astype("float32")
df["HDD"] = pd.to_numeric(df["HDD"], errors="coerce").fillna(0.0).astype("float32")
df["wind_speed"] = pd.to_numeric(df["wind_speed"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_2"] = pd.to_numeric(df["Temperature_lag_2"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_6"] = pd.to_numeric(df["Temperature_lag_6"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_48"] = pd.to_numeric(df["Temperature_lag_48"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_144"] = pd.to_numeric(df["Temperature_lag_144"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_2"] = pd.to_numeric(df["temperature_lag_2"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_6"] = pd.to_numeric(df["temperature_lag_6"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_48"] = pd.to_numeric(df["temperature_lag_48"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_144"] = pd.to_numeric(df["temperature_lag_144"], errors="coerce").fillna(0.0).astype("float32")
df["kWh"] = pd.to_numeric(df["kWh"], errors="coerce").fillna(0.0).astype("float32")

# CDD/HDD are simple weather stress features: cooling demand above 18C and heating demand below 18C.
temp = pd.to_numeric(df["Temperature"], errors="coerce").fillna(0.0)
df["CDD"] = np.maximum(temp - 18.0, 0.0).astype("float32")
df["HDD"] = np.maximum(18.0 - temp, 0.0).astype("float32")

# Encode the two static categorical fields as integer IDs for embedding layers.
df["StationNo"] = df["StationNo"].astype("string").fillna("missing")
df["StationNo_id"], _ = pd.factorize(df["StationNo"], sort=True)
df["StationNo_id"] = df["StationNo_id"].astype("int64")
df["TRIAL_REGION_NAME"] = df["TRIAL_REGION_NAME"].astype("string").fillna("missing")
df["TRIAL_REGION_NAME_id"], _ = pd.factorize(df["TRIAL_REGION_NAME"], sort=True)
df["TRIAL_REGION_NAME_id"] = df["TRIAL_REGION_NAME_id"].astype("int64")

num_stations = int(df["StationNo_id"].max()) + 1
num_regions = int(df["TRIAL_REGION_NAME_id"].max()) + 1

# Resample each customer to FREQ. kWh is summed over the bin; static fields use the first value; weather uses the mean.
agg = {
    TARGET_COL: "sum",
    "NUM_OCCUPANTS": "first",
    "NUM_ROOMS_HEATED": "first",
    "NUM_REFRIGERATORS": "first",
    "Unit": "first",
    "SemiDetached": "first",
    "SeparateHouse": "first",
    "HAS_GAS_HEATING": "first",
    "HAS_GAS_HOT_WATER": "first",
    "HAS_GAS_COOKING": "first",
    "HAS_POOLPUMP": "first",
    "Ducted": "first",
    "SplitSystem": "first",
    "NoAirCon": "first",
    "OtherAirCon": "first",
    "CONTROLLED_LOAD_CNT": "first",
    "StationNo_id": "first",
    "TRIAL_REGION_NAME_id": "first",
    "Temperature": "mean",
    "Weekday": "mean",
    "Month": "mean",
    "Hour": "mean",
    "CDD": "mean",
    "HDD": "mean",
    "wind_speed": "mean",
    "Temperature_lag_2": "mean",
    "Temperature_lag_6": "mean",
    "Temperature_lag_48": "mean",
    "Temperature_lag_144": "mean",
    "temperature_lag_2": "mean",
    "temperature_lag_6": "mean",
    "temperature_lag_48": "mean",
    "temperature_lag_144": "mean",
}
df = (
    df.set_index(TIME_COL)
      .groupby(HOUSEHOLD_ID_COL)
      .resample(pandas_freq(FREQ))
      .agg(agg)
      .reset_index()
      .sort_values([HOUSEHOLD_ID_COL, TIME_COL])
)

# Final numeric cleanup after resampling.
df["NUM_OCCUPANTS"] = pd.to_numeric(df["NUM_OCCUPANTS"], errors="coerce").fillna(0.0).astype("float32")
df["NUM_ROOMS_HEATED"] = pd.to_numeric(df["NUM_ROOMS_HEATED"], errors="coerce").fillna(0.0).astype("float32")
df["NUM_REFRIGERATORS"] = pd.to_numeric(df["NUM_REFRIGERATORS"], errors="coerce").fillna(0.0).astype("float32")
df["Unit"] = pd.to_numeric(df["Unit"], errors="coerce").fillna(0.0).astype("float32")
df["SemiDetached"] = pd.to_numeric(df["SemiDetached"], errors="coerce").fillna(0.0).astype("float32")
df["SeparateHouse"] = pd.to_numeric(df["SeparateHouse"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_GAS_HEATING"] = pd.to_numeric(df["HAS_GAS_HEATING"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_GAS_HOT_WATER"] = pd.to_numeric(df["HAS_GAS_HOT_WATER"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_GAS_COOKING"] = pd.to_numeric(df["HAS_GAS_COOKING"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_POOLPUMP"] = pd.to_numeric(df["HAS_POOLPUMP"], errors="coerce").fillna(0.0).astype("float32")
df["Ducted"] = pd.to_numeric(df["Ducted"], errors="coerce").fillna(0.0).astype("float32")
df["SplitSystem"] = pd.to_numeric(df["SplitSystem"], errors="coerce").fillna(0.0).astype("float32")
df["NoAirCon"] = pd.to_numeric(df["NoAirCon"], errors="coerce").fillna(0.0).astype("float32")
df["OtherAirCon"] = pd.to_numeric(df["OtherAirCon"], errors="coerce").fillna(0.0).astype("float32")
df["CONTROLLED_LOAD_CNT"] = pd.to_numeric(df["CONTROLLED_LOAD_CNT"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature"] = pd.to_numeric(df["Temperature"], errors="coerce").fillna(0.0).astype("float32")
df["CDD"] = pd.to_numeric(df["CDD"], errors="coerce").fillna(0.0).astype("float32")
df["HDD"] = pd.to_numeric(df["HDD"], errors="coerce").fillna(0.0).astype("float32")
df["wind_speed"] = pd.to_numeric(df["wind_speed"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_2"] = pd.to_numeric(df["Temperature_lag_2"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_6"] = pd.to_numeric(df["Temperature_lag_6"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_48"] = pd.to_numeric(df["Temperature_lag_48"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_144"] = pd.to_numeric(df["Temperature_lag_144"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_2"] = pd.to_numeric(df["temperature_lag_2"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_6"] = pd.to_numeric(df["temperature_lag_6"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_48"] = pd.to_numeric(df["temperature_lag_48"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_144"] = pd.to_numeric(df["temperature_lag_144"], errors="coerce").fillna(0.0).astype("float32")
df["kWh"] = pd.to_numeric(df["kWh"], errors="coerce").fillna(0.0).astype("float32")
df["StationNo_id"] = pd.to_numeric(df["StationNo_id"], errors="coerce").ffill().bfill().fillna(0).astype("int64")
df["TRIAL_REGION_NAME_id"] = pd.to_numeric(df["TRIAL_REGION_NAME_id"], errors="coerce").ffill().bfill().fillna(0).astype("int64")

print("Preprocessed dataframe shape:", df.shape)
print("Static category counts:", "stations", num_stations, "| regions", num_regions)


In [ ]:

# Use a seeded random generator so the customer split is repeatable.
rng = np.random.default_rng(SEED)
# Split by unique customers, not by rows/windows, to avoid leakage.
all_customers = np.array(sorted(df[HOUSEHOLD_ID_COL].dropna().unique()))
n_val = max(1, int(len(all_customers) * VAL_FRAC))
val_customers = set(rng.choice(all_customers, size=n_val, replace=False).tolist())
print(f"Customers total={len(all_customers)} | train={len(all_customers)-len(val_customers)} | val={len(val_customers)}")

def empty_float(*shape):
    return np.empty(shape, dtype=np.float32)

def empty_int(*shape):
    return np.empty(shape, dtype=np.int64)

# Accumulate train and validation windows before stacking them into tensors.
y_past_parts, y_future_parts = [], []
x_time_past_parts, x_time_future_parts = [], []
x_static_parts, x_static_cat_parts = [], []
y_past_val_parts, y_future_val_parts = [], []
x_time_past_val_parts, x_time_future_val_parts = [], []
x_static_val_parts, x_static_cat_val_parts = [], []

# Process one customer at a time so windows never cross household boundaries.
for household_id, g in df.groupby(HOUSEHOLD_ID_COL):
    g = g.sort_values(TIME_COL).drop_duplicates(subset=[TIME_COL], keep="last")
    if g.empty:
        continue

    # Rebuild a complete regular timestamp index for this customer.
    idx = pd.date_range(g[TIME_COL].min(), g[TIME_COL].max(), freq=pandas_freq(FREQ))
    g = g.set_index(TIME_COL).reindex(idx)
    g.index.name = TIME_COL
    g[HOUSEHOLD_ID_COL] = household_id

    # Fill static customer fields explicitly. These should be constant within a household.
    g["NUM_OCCUPANTS"] = pd.to_numeric(g["NUM_OCCUPANTS"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["NUM_ROOMS_HEATED"] = pd.to_numeric(g["NUM_ROOMS_HEATED"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["NUM_REFRIGERATORS"] = pd.to_numeric(g["NUM_REFRIGERATORS"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["Unit"] = pd.to_numeric(g["Unit"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["SemiDetached"] = pd.to_numeric(g["SemiDetached"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["SeparateHouse"] = pd.to_numeric(g["SeparateHouse"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["HAS_GAS_HEATING"] = pd.to_numeric(g["HAS_GAS_HEATING"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["HAS_GAS_HOT_WATER"] = pd.to_numeric(g["HAS_GAS_HOT_WATER"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["HAS_GAS_COOKING"] = pd.to_numeric(g["HAS_GAS_COOKING"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["HAS_POOLPUMP"] = pd.to_numeric(g["HAS_POOLPUMP"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["Ducted"] = pd.to_numeric(g["Ducted"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["SplitSystem"] = pd.to_numeric(g["SplitSystem"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["NoAirCon"] = pd.to_numeric(g["NoAirCon"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["OtherAirCon"] = pd.to_numeric(g["OtherAirCon"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["CONTROLLED_LOAD_CNT"] = pd.to_numeric(g["CONTROLLED_LOAD_CNT"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["StationNo_id"] = pd.to_numeric(g["StationNo_id"], errors="coerce").ffill().bfill().fillna(0).astype("int64")
    g["TRIAL_REGION_NAME_id"] = pd.to_numeric(g["TRIAL_REGION_NAME_id"], errors="coerce").ffill().bfill().fillna(0).astype("int64")

    # Add cyclical features once on the final per-customer timestamp grid.
    add_cyclical_time_features(g, g.index)

    # Fill target and time-varying covariates explicitly.
    g["kWh"] = pd.to_numeric(g["kWh"], errors="coerce").fillna(0.0).astype("float32")
    g["Temperature"] = pd.to_numeric(g["Temperature"], errors="coerce").fillna(0.0).astype("float32")
    g["Hour_sin"] = pd.to_numeric(g["Hour_sin"], errors="coerce").fillna(0.0).astype("float32")
    g["Hour_cos"] = pd.to_numeric(g["Hour_cos"], errors="coerce").fillna(0.0).astype("float32")
    g["Weekday_sin"] = pd.to_numeric(g["Weekday_sin"], errors="coerce").fillna(0.0).astype("float32")
    g["Weekday_cos"] = pd.to_numeric(g["Weekday_cos"], errors="coerce").fillna(0.0).astype("float32")
    g["Month_sin"] = pd.to_numeric(g["Month_sin"], errors="coerce").fillna(0.0).astype("float32")
    g["Month_cos"] = pd.to_numeric(g["Month_cos"], errors="coerce").fillna(0.0).astype("float32")
    g["CDD"] = pd.to_numeric(g["CDD"], errors="coerce").fillna(0.0).astype("float32")
    g["HDD"] = pd.to_numeric(g["HDD"], errors="coerce").fillna(0.0).astype("float32")
    g["wind_speed"] = pd.to_numeric(g["wind_speed"], errors="coerce").fillna(0.0).astype("float32")
    g["Temperature_lag_2"] = pd.to_numeric(g["Temperature_lag_2"], errors="coerce").fillna(0.0).astype("float32")
    g["Temperature_lag_6"] = pd.to_numeric(g["Temperature_lag_6"], errors="coerce").fillna(0.0).astype("float32")
    g["Temperature_lag_48"] = pd.to_numeric(g["Temperature_lag_48"], errors="coerce").fillna(0.0).astype("float32")
    g["Temperature_lag_144"] = pd.to_numeric(g["Temperature_lag_144"], errors="coerce").fillna(0.0).astype("float32")
    g["temperature_lag_2"] = pd.to_numeric(g["temperature_lag_2"], errors="coerce").fillna(0.0).astype("float32")
    g["temperature_lag_6"] = pd.to_numeric(g["temperature_lag_6"], errors="coerce").fillna(0.0).astype("float32")
    g["temperature_lag_48"] = pd.to_numeric(g["temperature_lag_48"], errors="coerce").fillna(0.0).astype("float32")
    g["temperature_lag_144"] = pd.to_numeric(g["temperature_lag_144"], errors="coerce").fillna(0.0).astype("float32")

    g = g.reset_index()
    if len(g) < CONTEXT_LEN + SEQ_LEN:
        continue

    # Convert this customer's cleaned dataframe into arrays for the sliding-window loop.
    y_all = g[[TARGET_COL]].to_numpy(dtype=np.float32)
    x_time_past_all = g[PAST_COV_COLS].to_numpy(dtype=np.float32)
    x_time_future_all = g[FUTURE_COV_COLS].to_numpy(dtype=np.float32)
    x_static_vec = g[STATIC_COLS].iloc[0].to_numpy(dtype=np.float32)
    x_cat_vec = g[["StationNo_id", "TRIAL_REGION_NAME_id"]].iloc[0].to_numpy(dtype=np.int64)

    # Every window from this household goes to either train or validation, never both.
    is_val = household_id in val_customers
    yp_list = y_past_val_parts if is_val else y_past_parts
    yf_list = y_future_val_parts if is_val else y_future_parts
    xtp_list = x_time_past_val_parts if is_val else x_time_past_parts
    xtf_list = x_time_future_val_parts if is_val else x_time_future_parts
    xs_list = x_static_val_parts if is_val else x_static_parts
    xc_list = x_static_cat_val_parts if is_val else x_static_cat_parts

    # Slide the manually selected context/target window across this customer.
    max_start = len(g) - CONTEXT_LEN - SEQ_LEN
    for start in range(0, max_start + 1, WINDOW_STRIDE):
        mid = start + CONTEXT_LEN
        end = mid + SEQ_LEN
        yp_list.append(y_all[start:mid])
        yf_list.append(y_all[mid:end])
        xtp_list.append(x_time_past_all[start:mid])
        xtf_list.append(x_time_future_all[mid:end])
        xs_list.append(x_static_vec)
        xc_list.append(x_cat_vec)

# Stack collected lists into dense arrays with model-ready dtypes.
y_past = np.stack(y_past_parts).astype(np.float32)
y_future = np.stack(y_future_parts).astype(np.float32)
x_time_past = np.stack(x_time_past_parts).astype(np.float32)
x_time_future = np.stack(x_time_future_parts).astype(np.float32)
x_static = np.stack(x_static_parts).astype(np.float32)
static_cat_ids = np.stack(x_static_cat_parts).astype(np.int64)

y_past_val = np.stack(y_past_val_parts).astype(np.float32)
y_future_val = np.stack(y_future_val_parts).astype(np.float32)
x_time_past_val = np.stack(x_time_past_val_parts).astype(np.float32)
x_time_future_val = np.stack(x_time_future_val_parts).astype(np.float32)
x_static_val = np.stack(x_static_val_parts).astype(np.float32)
static_cat_ids_val = np.stack(x_static_cat_val_parts).astype(np.int64)

print("TRAIN y_past:", y_past.shape, "y_future:", y_future.shape, "x_time_past:", x_time_past.shape, "x_time_future:", x_time_future.shape)
print("VAL   y_past:", y_past_val.shape, "y_future:", y_future_val.shape, "x_time_past:", x_time_past_val.shape, "x_time_future:", x_time_future_val.shape)

# Fit target scaling on training kWh only, then reuse it for validation and inverse transforms.
y_scaler = MinMaxScaler(feature_range=(0, 1))
y_train_log_flat = np.concatenate([
    np.log1p(y_past).reshape(-1, 1),
    np.log1p(y_future).reshape(-1, 1),
], axis=0)
y_scaler.fit(y_train_log_flat)

def scale_y(arr):
    if np.asarray(arr).size == 0:
        return np.asarray(arr, dtype=np.float32)
    return y_scaler.transform(np.log1p(arr).reshape(-1, 1)).reshape(arr.shape).astype(np.float32)

def y_scaled_to_kwh(arr_scaled):
    arr_log = y_scaler.inverse_transform(np.asarray(arr_scaled).reshape(-1, 1)).reshape(np.asarray(arr_scaled).shape)
    return np.maximum(np.expm1(arr_log), 0.0).astype(np.float32)

y_past_scaled = scale_y(y_past)
y_future_scaled = scale_y(y_future)
y_past_scaled_val = scale_y(y_past_val)
y_future_scaled_val = scale_y(y_future_val)

y_dim = y_future_scaled.shape[-1]
past_c_dim = x_time_past.shape[-1]
future_c_dim = x_time_future.shape[-1]
s_dim = x_static.shape[-1]

# Fit past and future covariate scalers separately so unknowable weather never leaks into the future branch.
past_c_scaler = StandardScaler()
future_c_scaler = StandardScaler()

if x_time_past.size:
    past_c_scaler.fit(x_time_past.reshape(-1, past_c_dim))
if x_time_future.size:
    future_c_scaler.fit(x_time_future.reshape(-1, future_c_dim))

def scale_time_covariates(arr, scaler, dim):
    if arr.size == 0:
        return arr.astype(np.float32)
    return scaler.transform(arr.reshape(-1, dim)).reshape(arr.shape).astype(np.float32)

x_time_past_scaled = scale_time_covariates(x_time_past, past_c_scaler, past_c_dim)
x_time_past_scaled_val = scale_time_covariates(x_time_past_val, past_c_scaler, past_c_dim)
x_time_future_scaled = scale_time_covariates(x_time_future, future_c_scaler, future_c_dim)
x_time_future_scaled_val = scale_time_covariates(x_time_future_val, future_c_scaler, future_c_dim)

# Fit static numeric scaling on training windows only.
s_scaler = StandardScaler()
x_static_scaled = s_scaler.fit_transform(x_static).astype(np.float32)
x_static_scaled_val = s_scaler.transform(x_static_val).astype(np.float32)

if CONTEXT_LEN > 0:
    print("Scaled kWh range check:", y_past_scaled.min(), y_past_scaled.max(), y_future_scaled.min(), y_future_scaled.max())
else:
    print("Scaled kWh range check:", y_future_scaled.min(), y_future_scaled.max())
print("Model input dimensions:", "y_dim", y_dim, "past_c_dim", past_c_dim, "future_c_dim", future_c_dim, "s_dim", s_dim)


# Wrap the prepared NumPy windows in a PyTorch Dataset so the training loop receives consistent minibatches.
class ForecastWindowDataset(Dataset):
    def __init__(self, y_past, y_future, x_time_past, x_time_future, x_static, x_static_cat):
        self.y_past = torch.from_numpy(y_past).float()
        self.y_future = torch.from_numpy(y_future).float()
        self.x_time_past = torch.from_numpy(x_time_past).float()
        self.x_time_future = torch.from_numpy(x_time_future).float()
        self.x_static = torch.from_numpy(x_static).float()
        self.x_static_cat = torch.from_numpy(x_static_cat).long()

    def __len__(self):
        return self.y_future.shape[0]

    def __getitem__(self, idx):
        return (
            self.y_past[idx],
            self.y_future[idx],
            self.x_time_past[idx],
            self.x_time_future[idx],
            self.x_static[idx],
            self.x_static_cat[idx],
        )

train_dataset = ForecastWindowDataset(
    y_past_scaled,
    y_future_scaled,
    x_time_past_scaled,
    x_time_future_scaled,
    x_static_scaled,
    static_cat_ids,
)
loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
print("Training windows:", len(train_dataset), "| batches per epoch:", len(loader))


In [ ]:

# Sinusoidal positional encoding follows the Transformer paper; source: https://arxiv.org/abs/1706.03762
# Implementation reference for PyTorch Transformer/RNN sequence modelling: https://github.com/pytorch/examples/tree/main/word_language_model
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term[:pe[:, 1::2].shape[1]])
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]
# Combines future time covariates, static numeric features, categorical embeddings, and optional past context.
class FutureConditionProjector(nn.Module):
    def __init__(self, future_c_dim, s_num_dim, past_dim, cond_dim, num_stations, num_regions, station_emb_dim=16, region_emb_dim=4):
        super().__init__()
        self.station_emb = nn.Embedding(num_stations, station_emb_dim)
        self.region_emb = nn.Embedding(num_regions, region_emb_dim)
        in_dim = future_c_dim + s_num_dim + station_emb_dim + region_emb_dim + past_dim
        self.proj = nn.Sequential(
            nn.Linear(in_dim, cond_dim),
            nn.LayerNorm(cond_dim),
            nn.Tanh(),
        )

    def forward(self, x_time_future, x_static_num, x_static_cat, past_context):
        B, T, _ = x_time_future.shape
        station_id = x_static_cat[:, 0].long()
        region_id = x_static_cat[:, 1].long()
        st = self.station_emb(station_id).unsqueeze(1).expand(B, T, -1)
        rg = self.region_emb(region_id).unsqueeze(1).expand(B, T, -1)
        xs = x_static_num.unsqueeze(1).expand(B, T, -1)
        pc = past_context.unsqueeze(1).expand(B, T, -1)
        return self.proj(torch.cat([x_time_future, xs, st, rg, pc], dim=-1))
# Gaussian negative log-likelihood trains mean and scale forecasts; source: https://pytorch.org/docs/stable/distributions.html#normal
def gaussian_nll(y, mean, log_scale):
    inv_var = torch.exp(-2.0 * log_scale)
    return 0.5 * ((y - mean) ** 2 * inv_var + 2.0 * log_scale + math.log(2.0 * math.pi))


In [ ]:

# Causal transformer outputs a probabilistic distribution one future step at a time; source: https://arxiv.org/abs/1706.03762
# Implementation reference for minimal causal GPT-style Transformer code: https://github.com/karpathy/minGPT
# Additional PyTorch sequence-model implementation reference: https://github.com/pytorch/examples/tree/main/word_language_model
class ForecastAutoregressiveTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.cond = FutureConditionProjector(future_c_dim, s_dim, hidden_dim, cond_dim, num_stations, num_regions)
        self.input_proj = nn.Linear(y_dim + cond_dim, hidden_dim)
        self.pos = SinusoidalPositionalEncoding(hidden_dim, SEQ_LEN)
        # PyTorch TransformerEncoderLayer supplies the attention block used by transformer variants; implementation: https://github.com/pytorch/pytorch/tree/main/torch/nn/modules/transformer.py
        layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=transformer_heads, dim_feedforward=transformer_ff_dim,
            dropout=transformer_dropout, activation="gelu", batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=transformer_layers)
        self.norm = nn.LayerNorm(hidden_dim)
        self.mean = nn.Linear(hidden_dim, y_dim)
        self.log_scale = nn.Linear(hidden_dim, y_dim)

    # Causal mask prevents the autoregressive transformer from seeing future target steps; source: https://arxiv.org/abs/1706.03762
    # Implementation reference for causal Transformer masking/generation: https://github.com/karpathy/minGPT
    def causal_mask(self, T, device):
        return torch.triu(torch.ones(T, T, device=device, dtype=torch.bool), diagonal=1)

    # Builds the conditional tensor reused by training, sampling, and evaluation.
    def build_cond(self, y_past, x_time_past, x_time_future, x_static_num, x_static_cat):
        past_context = torch.zeros(y_past.shape[0], hidden_dim, device=y_past.device, dtype=y_past.dtype)
        cond = self.cond(x_time_future, x_static_num, x_static_cat, past_context)
        return cond

    def forward(self, y_future, y_past, x_time_past, x_time_future, x_static_num, x_static_cat):
        cond = self.build_cond(y_past, x_time_past, x_time_future, x_static_num, x_static_cat)
        if y_past.shape[1] == 0:
            start = torch.zeros(y_future.shape[0], 1, y_future.shape[-1], device=y_future.device, dtype=y_future.dtype)
        else:
            start = y_past[:, -1:, :]
        shifted = torch.cat([start, y_future[:, :-1, :]], dim=1)
        h = self.input_proj(torch.cat([shifted, cond], dim=-1))
        h = self.transformer(self.pos(h), mask=self.causal_mask(h.shape[1], h.device))
        h = self.norm(h)
        mean = torch.sigmoid(self.mean(h))
        log_scale = self.log_scale(h).clamp(min_log_scale, max_log_scale)
        return mean, log_scale

    # torch.no_grad disables autograd during sampling/evaluation to save memory; implementation: https://github.com/pytorch/pytorch
    @torch.no_grad()
    def sample(self, y_past, x_time_past, x_time_future, x_static_num, x_static_cat):
        self.eval()
        cond = self.build_cond(y_past, x_time_past, x_time_future, x_static_num, x_static_cat)
        generated = []
        if y_past.shape[1] == 0:
            prev_full = torch.zeros(x_time_future.shape[0], 1, y_dim, device=x_time_future.device, dtype=x_time_future.dtype)
        else:
            prev_full = y_past[:, -1:, :]
        for t in range(SEQ_LEN):
            if generated:
                teacher = torch.cat([prev_full, torch.cat(generated, dim=1)], dim=1)[:, :t+1, :]
            else:
                teacher = prev_full
            cond_now = cond[:, :t+1, :]
            h = self.input_proj(torch.cat([teacher, cond_now], dim=-1))
            h = self.transformer(self.pos(h), mask=self.causal_mask(h.shape[1], h.device))
            h = self.norm(h[:, -1:, :])
            mean = torch.sigmoid(self.mean(h))
            log_scale = self.log_scale(h).clamp(min_log_scale, max_log_scale)
            y_next = (mean + torch.exp(log_scale) * torch.randn_like(mean) * sample_temperature).clamp(0.0, 1.0)
            generated.append(y_next)
        return torch.cat(generated, dim=1)

model_ar = ForecastAutoregressiveTransformer().to(DEVICE)
# AdamW is the decoupled-weight-decay optimiser used for neural training; implementation: https://github.com/pytorch/pytorch/tree/main/torch/optim
opt_ar = optim.AdamW(model_ar.parameters(), lr=LR, weight_decay=1e-4)

for epoch in range(epochs_ar):
    model_ar.train()
    running = running_nll = 0.0
    for y_past_b, y_future_b, x_time_past_b, x_time_future_b, x_static_b, x_static_cat_b in loader:
        y_past_b = y_past_b.to(DEVICE, non_blocking=True)
        y_future_b = y_future_b.to(DEVICE, non_blocking=True)
        x_time_past_b = x_time_past_b.to(DEVICE, non_blocking=True)
        x_time_future_b = x_time_future_b.to(DEVICE, non_blocking=True)
        x_static_b = x_static_b.to(DEVICE, non_blocking=True)
        x_static_cat_b = x_static_cat_b.to(DEVICE, non_blocking=True).long()
        mean, log_scale = model_ar(y_future_b, y_past_b, x_time_past_b, x_time_future_b, x_static_b, x_static_cat_b)
        nll_per = gaussian_nll(y_future_b, mean, log_scale)
        loss_nll = nll_per.mean()
        loss = loss_nll
        opt_ar.zero_grad(set_to_none=True)
        loss.backward()
        if grad_clip:
            # Gradient clipping is a standard stabilisation utility for RNN/Transformer/GAN training; implementation: https://github.com/pytorch/pytorch/tree/main/torch/nn/utils
            torch.nn.utils.clip_grad_norm_(model_ar.parameters(), grad_clip)
        opt_ar.step()
        running += loss.item()
        running_nll += loss_nll.item()
    if epoch % 10 == 0:
        print(f"[{MODEL_LABEL}] epoch {epoch:03d} | loss={running/len(loader):.4f} | nll={running_nll/len(loader):.4f}")

active_model = model_ar

# torch.no_grad disables autograd during sampling/evaluation to save memory; implementation: https://github.com/pytorch/pytorch
@torch.no_grad()
# Generates one batch of probabilistic forecasts in scaled space before kWh inversion.
def generate_one_batch(y_past_b, x_time_past_b, x_time_future_b, x_static_b, x_static_cat_b):
    return active_model.sample(y_past_b, x_time_past_b, x_time_future_b, x_static_b, x_static_cat_b)


In [ ]:

# Saves enough metadata with the weights to identify the exact experiment settings later.
def checkpoint_payload():
    extra = {
        "model_type": MODEL_LABEL,
        "FREQ": FREQ,
        "PRIMARY_HORIZON": PRIMARY_HORIZON,
        "SEQ_LEN": SEQ_LEN,
        "CONTEXT_LEN": CONTEXT_LEN,
        "WINDOW_STRIDE": WINDOW_STRIDE,
        "PAST_COV_COLS": PAST_COV_COLS,
        "FUTURE_COV_COLS": FUTURE_COV_COLS,
        "STATIC_COLS": STATIC_COLS,
        "CATEGORICAL_COLS": CATEGORICAL_COLS,
        "y_scaler_min": y_scaler.min_.tolist(),
        "y_scaler_scale": y_scaler.scale_.tolist(),
        "num_stations": num_stations,
        "num_regions": num_regions,
        "hidden_dim": hidden_dim,
        "cond_dim": cond_dim,
        "latent_dim": latent_dim,
    }
    return extra

checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)
checkpoint_name = (
    f"{MODEL_TAG}__freq-{FREQ}__hor-{PRIMARY_HORIZON}__ctx-{CONTEXT_LEN}__seq-{SEQ_LEN}"
    f"__seed-{SEED}"
)
checkpoint_path = checkpoint_dir / f"{checkpoint_name}.pt"

payload = {"extra": checkpoint_payload(), "model_state_dict": model_ar.state_dict(), "optimizer_state_dict": opt_ar.state_dict()}
# torch.save serialises model weights plus metadata for reproducible checkpoint loading; implementation: https://github.com/pytorch/pytorch
torch.save(payload, checkpoint_path)
print("Saved:", checkpoint_path)


In [ ]:

SAMPLES = 200
EVAL_BATCH_SIZE = 64
EVAL_SEED = SEED + 123
EVAL_POOLS = ("validation",)
BINS = 80
CLIP_QUANTILE = None
ALPHA_PI = 0.10

# Dynamic Time Warping compares sequence shape even when peaks are slightly shifted; source: https://en.wikipedia.org/wiki/Dynamic_time_warping
def dtw_distance(a, b):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    n, m = len(a), len(b)
    dp = np.full((n + 1, m + 1), np.inf, dtype=np.float32)
    dp[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = abs(a[i - 1] - b[j - 1])
            dp[i, j] = cost + min(dp[i - 1, j], dp[i, j - 1], dp[i - 1, j - 1])
    return float(dp[n, m])

# Autocorrelation checks whether generated windows preserve temporal dependence; source: https://en.wikipedia.org/wiki/Autocorrelation
def autocorr(x, max_lag):
    x = np.asarray(x, dtype=np.float32)
    x = x - x.mean()
    denom = np.dot(x, x) + 1e-12
    out = [1.0]
    for lag in range(1, max_lag + 1):
        out.append(float(np.dot(x[:-lag], x[lag:]) / denom))
    return np.asarray(out)

def mean_acf(arr):
    max_lag = min(200, arr.shape[1] - 1)
    return np.mean([autocorr(arr[i, :, 0], max_lag) for i in range(arr.shape[0])], axis=0)

def window_feature_embed(x):
    y = x[:, :, 0]
    acf_lag = min(3, y.shape[1] - 1)
    acfs = np.stack([autocorr(row, acf_lag) for row in y], axis=0)
    return np.column_stack([
        y.mean(axis=1),
        y.std(axis=1),
        y.min(axis=1),
        y.max(axis=1),
        # NumPy quantile forms median/interval forecasts and QQ-plot points; codebase: https://github.com/numpy/numpy
        np.quantile(y, 0.10, axis=1),
        # NumPy quantile forms median/interval forecasts and QQ-plot points; codebase: https://github.com/numpy/numpy
        np.quantile(y, 0.50, axis=1),
        # NumPy quantile forms median/interval forecasts and QQ-plot points; codebase: https://github.com/numpy/numpy
        np.quantile(y, 0.90, axis=1),
        y.sum(axis=1),
        y.argmax(axis=1) / max(1, y.shape[1] - 1),
        acfs[:, 1] if acfs.shape[1] > 1 else np.zeros(y.shape[0]),
        acfs[:, 2] if acfs.shape[1] > 2 else np.zeros(y.shape[0]),
        acfs[:, 3] if acfs.shape[1] > 3 else np.zeros(y.shape[0]),
    ]).astype(np.float64)

def sqrtm_psd(mat):
    # Eigenvalue square-root is used for the Fréchet covariance term; SciPy sqrtm is the full reference: https://github.com/scipy/scipy/blob/main/scipy/linalg/_matfuncs_sqrtm.py
    vals, vecs = np.linalg.eigh((mat + mat.T) / 2.0)
    vals = np.clip(vals, 0.0, None)
    return (vecs * np.sqrt(vals)) @ vecs.T

# Feature Fréchet distance mirrors FID-style Gaussian distance on summary features; source: https://arxiv.org/abs/1706.08500
def frechet_distance(a, b):
    ma, mb = a.mean(axis=0), b.mean(axis=0)
    ca = np.cov(a, rowvar=False) + np.eye(a.shape[1]) * 1e-6
    cb = np.cov(b, rowvar=False) + np.eye(b.shape[1]) * 1e-6
    sqrt_ca = sqrtm_psd(ca)
    covmean = sqrtm_psd(sqrt_ca @ cb @ sqrt_ca)
    return float(np.sum((ma - mb) ** 2) + np.trace(ca + cb - 2 * covmean))

# Pinball loss evaluates quantile forecasts; source: https://en.wikipedia.org/wiki/Quantile_regression
def pinball(y, qhat, q):
    err = y - qhat
    return np.maximum(q * err, (q - 1) * err)

def run_eval_pool(EVAL_POOL):
    rng_eval = np.random.default_rng(EVAL_SEED + (0 if EVAL_POOL == "train" else 10_000))
    if EVAL_POOL == "train":
        n = min(512, len(y_future))
        idx = rng_eval.choice(len(y_future), size=n, replace=False)
        y_past_in = y_past_scaled[idx]
        x_time_past_in = x_time_past_scaled[idx]
        x_time_future_in = x_time_future_scaled[idx]
        x_static_in = x_static_scaled[idx]
        x_static_cat_in = static_cat_ids[idx]
        y_real_kwh = y_future[idx]
    elif EVAL_POOL == "validation":
        n = min(512, len(y_future_val))
        idx = rng_eval.choice(len(y_future_val), size=n, replace=False)
        y_past_in = y_past_scaled_val[idx]
        x_time_past_in = x_time_past_scaled_val[idx]
        x_time_future_in = x_time_future_scaled_val[idx]
        x_static_in = x_static_scaled_val[idx]
        x_static_cat_in = static_cat_ids_val[idx]
        y_real_kwh = y_future_val[idx]
    else:
        raise ValueError(EVAL_POOL)

    PLOT_N = len(y_real_kwh)
    selected_train_windows = PLOT_N if EVAL_POOL == "train" else 0
    selected_val_windows = PLOT_N if EVAL_POOL == "validation" else 0
    DTW_N = min(128, PLOT_N)
    print(f"eval_pool={EVAL_POOL} | windows={PLOT_N} | samples={SAMPLES}")

    y_fake_scaled_samples = []
    active_model.eval()
    # torch.no_grad disables autograd during sampling/evaluation to save memory; implementation: https://github.com/pytorch/pytorch
    with torch.no_grad():
        for s in range(SAMPLES):
            if s % 10 == 0:
                print(f"Generating {MODEL_LABEL} sample {s+1}/{SAMPLES}")
            batches = []
            for start in range(0, PLOT_N, EVAL_BATCH_SIZE):
                end = min(start + EVAL_BATCH_SIZE, PLOT_N)
                ypb = torch.from_numpy(y_past_in[start:end]).to(DEVICE)
                xtpb = torch.from_numpy(x_time_past_in[start:end]).to(DEVICE)
                xtfb = torch.from_numpy(x_time_future_in[start:end]).to(DEVICE)
                xsb = torch.from_numpy(x_static_in[start:end]).to(DEVICE)
                xcb = torch.from_numpy(x_static_cat_in[start:end]).to(DEVICE).long()
                batches.append(generate_one_batch(ypb, xtpb, xtfb, xsb, xcb).cpu().numpy())
            y_fake_scaled_samples.append(np.concatenate(batches, axis=0))

    y_fake_scaled_samples = np.stack(y_fake_scaled_samples, axis=0)
    y_fake_kwh_samples = np.stack([y_scaled_to_kwh(y_fake_scaled_samples[s]) for s in range(SAMPLES)], axis=0)
    y_fake_kwh_point = np.median(y_fake_kwh_samples, axis=0)
    y_fake_kwh_mean = np.mean(y_fake_kwh_samples, axis=0)

    MAE_median = float(np.mean(np.abs(y_fake_kwh_point - y_real_kwh)))
    RMSE_mean = float(np.sqrt(np.mean((y_fake_kwh_mean - y_real_kwh) ** 2)))
    PeakMAE = float(np.mean(np.abs(y_fake_kwh_point.max(axis=1) - y_real_kwh.max(axis=1))))

    real_flat = y_real_kwh.reshape(-1)
    fake_flat = y_fake_kwh_point.reshape(-1)
    lo, hi = float(real_flat.min()), float(real_flat.max())
    if hi <= lo:
        hi = lo + 1e-6
    edges = np.linspace(lo, hi, BINS + 1)
    # NumPy histogram underpins the marginal distribution/KL comparison; codebase: https://github.com/numpy/numpy
    p, _ = np.histogram(real_flat, bins=edges, density=True)
    # NumPy histogram underpins the marginal distribution/KL comparison; codebase: https://github.com/numpy/numpy
    q, _ = np.histogram(fake_flat, bins=edges, density=True)
    eps = 1e-8
    p = (p + eps) / (p + eps).sum()
    q = (q + eps) / (q + eps).sum()
    # KL histogram estimates marginal distribution mismatch from binned real/generated kWh; source: https://en.wikipedia.org/wiki/Kullback%E2%80%93Leibler_divergence
    KL_hist = float(np.sum(p * np.log(p / q)))

    dtw_idxs = rng_eval.choice(PLOT_N, size=DTW_N, replace=False)
    DTW_mean = float(np.mean([dtw_distance(y_real_kwh[i, :, 0], y_fake_kwh_point[i, :, 0]) for i in dtw_idxs]))

    FTSD = frechet_distance(window_feature_embed(y_real_kwh), window_feature_embed(y_fake_kwh_point))

    X = y_fake_kwh_samples[:, :, :, 0]
    Y = y_real_kwh[:, :, 0]
    # NumPy quantile forms median/interval forecasts and QQ-plot points; codebase: https://github.com/numpy/numpy
    q10 = np.quantile(X, 0.05, axis=0)
    # NumPy quantile forms median/interval forecasts and QQ-plot points; codebase: https://github.com/numpy/numpy
    q50 = np.quantile(X, 0.50, axis=0)
    # NumPy quantile forms median/interval forecasts and QQ-plot points; codebase: https://github.com/numpy/numpy
    q90 = np.quantile(X, 0.95, axis=0)
    # CRPS is approximated from quantile pinball losses; source: https://en.wikipedia.org/wiki/Continuous_ranked_probability_score
    CRPS = float(2.0 * np.mean([
        np.mean(pinball(Y, q10, 0.05)),
        np.mean(pinball(Y, q50, 0.50)),
        np.mean(pinball(Y, q90, 0.95)),
    ]))
    # QuantileLoss averages pinball losses at the requested forecast quantiles; source: https://en.wikipedia.org/wiki/Quantile_regression
    QuantileLoss = float(np.mean([
        np.mean(pinball(Y, q10, 0.05)),
        np.mean(pinball(Y, q50, 0.50)),
        np.mean(pinball(Y, q90, 0.95)),
    ]))
    width = q90 - q10
    below = Y < q10
    above = Y > q90
    # Winkler interval score rewards narrow prediction intervals but penalises missed coverage; source: https://epiforecasts.io/scoringutils/articles/scoring-rules.html
    Winkler = float(np.mean(width + (2 / ALPHA_PI) * (q10 - Y) * below + (2 / ALPHA_PI) * (Y - q90) * above))
    Coverage = float(np.mean((Y >= q10) & (Y <= q90)))
    ACF_error = float(np.mean(np.abs(mean_acf(y_real_kwh) - mean_acf(y_fake_kwh_point))))

    scale = float(np.mean(y_real_kwh) + 1e-6)
    peak_scale = float(np.mean(y_real_kwh.max(axis=1)) + 1e-6)
    # CompositeScore is a project-specific tuning heuristic, not a standard literature metric.
    CompositeScore = float(np.mean([
        MAE_median / scale,
        RMSE_mean / scale,
        PeakMAE / peak_scale,
        CRPS / scale,
        QuantileLoss / scale,
        Winkler / scale,
        ACF_error,
        KL_hist,
        DTW_mean / (SEQ_LEN * scale),
        min(FTSD, 1e6) / (1.0 + min(FTSD, 1e6)),
    ]))

    corr_real = np.nan
    corr_fake = np.nan
    # Future covariates intentionally exclude weather, so temperature-correlation diagnostics are disabled here.

    print("=" * 60)
    print(f"[{MODEL_LABEL}] forecast eval | pool={EVAL_POOL} | freq={FREQ} | horizon={PRIMARY_HORIZON} | context={CONTEXT_LEN} | seq_len={SEQ_LEN}")
    print(f"KL_hist={KL_hist:.6f} | DTW_mean={DTW_mean:.6f} | FTSD={FTSD:.6f}")
    print(f"MAE={MAE_median:.6f} | RMSE={RMSE_mean:.6f} | PeakMAE={PeakMAE:.6f}")
    print(f"CRPS={CRPS:.6f} | QuantileLoss={QuantileLoss:.6f} | Winkler={Winkler:.6f} | Coverage={Coverage:.4f}")
    print(f"ACF_error={ACF_error:.6f} | CompositeScore={CompositeScore:.6f}")

    return locals()

eval_results = {}
for _pool in EVAL_POOLS:
    eval_results[_pool] = run_eval_pool(_pool)

METRIC_NAMES = ['KL_hist', 'DTW_mean', 'FTSD', 'MAE_median', 'RMSE_mean', 'PeakMAE', 'CRPS', 'QuantileLoss', 'Winkler', 'Coverage', 'ACF_error', 'CompositeScore', 'corr_real', 'corr_fake']

def print_metrics_table(rows):
    table = pd.DataFrame(rows)
    cols = ["eval_pool"] + [m for m in METRIC_NAMES if m in table.columns]
    table = table[cols]
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(table.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
    return table

metric_rows = []
for eval_pool, namespace in eval_results.items():
    row = {"eval_pool": eval_pool}
    for metric_name in METRIC_NAMES:
        if metric_name in namespace:
            row[metric_name] = namespace[metric_name]
    metric_rows.append(row)
metrics_table = print_metrics_table(metric_rows)
